# L10 demo: a hyperparameter search, tracked and registered

We predict the Combined Cycle Power Plant's net output (MW) from four ambient
measurements, the same set as L9. The model is not the point. The point is that the
search that finds a good model is a machine for generating hundreds of runs, and
without a record of them you cannot say which run produced your number or reproduce it.

So we log every trial to **MLflow**, search with **Optuna**, register the winner, and
report a single honest test score at the end.

> Data: [UCI Combined Cycle Power Plant](https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant),
> 9,568 hourly records, carried over from L9.

## 1. Load CCPP and lock a test split

The test split is set aside now and scored exactly once, at the very end, on the single
model we select. Everything in between uses only the training portion.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CACHE = Path('data')
CACHE.mkdir(exist_ok=True)
URL = 'https://archive.ics.uci.edu/static/public/294/combined+cycle+power+plant.zip'
xlsx = CACHE / 'Folds5x2_pp.xlsx'
if not xlsx.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        archive = zipfile.ZipFile(io.BytesIO(r.read()))
    xlsx.write_bytes(archive.read('CCPP/Folds5x2_pp.xlsx'))

FEATURES, TARGET, SEED = ['AT', 'V', 'AP', 'RH'], 'PE', 0
df = pd.read_excel(xlsx, 'Sheet1')
X, y = df[FEATURES].to_numpy(), df[TARGET].to_numpy()
X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f'{len(X):,} rows; train {len(X_tr):,}, locked test {len(X_test):,}')

## 2. Search, with every trial tracked

The Optuna **study** proposes a configuration, we score it by 3-fold cross-validation on
the training data, and we log that trial to MLflow as a **nested run** under one parent
run for the whole study. Optuna's default sampler is TPE, which models the good regions
of the space and samples toward them.

The tracking backend is a local SQLite file, the default now that MLflow's bare file
store is deprecated.

In [ ]:
import mlflow
import optuna
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('ccpp-search')
cv = KFold(n_splits=3, shuffle=True, random_state=SEED)


def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
        max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 8, 128, log=True),
        max_iter=trial.suggest_int('max_iter', 50, 250),
        l2_regularization=trial.suggest_float('l2_regularization', 1e-6, 10.0, log=True),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 60),
    )
    model = HistGradientBoostingRegressor(random_state=SEED, **params)
    rmse = -cross_val_score(model, X_tr, y_tr, cv=cv, n_jobs=-1,
                            scoring='neg_root_mean_squared_error').mean()
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metric('val_rmse', rmse)
    return rmse


with mlflow.start_run(run_name='optuna-study') as parent:
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=25)
    mlflow.log_metric('best_val_rmse', study.best_value)
print(f'best validation RMSE {study.best_value:.3f} MW after {len(study.trials)} trials')

## 3. Read the runs back

Every trial is now a row in the tracking store. In practice you open the UI with
`mlflow ui` (or `mlflow server`) and sort; here we pull the same table with
`search_runs` and show the best few. Nothing about the search is hidden.

In [ ]:
runs = mlflow.search_runs(experiment_names=['ccpp-search'],
                          order_by=['metrics.val_rmse ASC'])
cols = ['metrics.val_rmse', 'params.learning_rate', 'params.max_leaf_nodes',
        'params.max_iter']
print(runs[runs['metrics.val_rmse'].notna()][cols].head(5).to_string(index=False))

Optuna can also say which hyperparameters actually moved the score. On this easy,
low-dimensional problem the answer is usually a single dominant knob, which is exactly
the Bergstra and Bengio point: most hyperparameters do not matter, and which one does
depends on the dataset.

In [ ]:
from optuna.importance import get_param_importances

importances = get_param_importances(study)
for name, imp in importances.items():
    print(f'{name:20s} {imp:.3f}')

## 4. Register the winner, with its lineage

Refit the best configuration on the full training set and register it under a name, so
it can be found again by URI rather than by remembering a run id. We log its data hash
and seed alongside, so the registered model carries its lineage.

In [ ]:
import hashlib

data_md5 = hashlib.md5(xlsx.read_bytes()).hexdigest()
winner = HistGradientBoostingRegressor(random_state=SEED, **study.best_params).fit(X_tr, y_tr)

with mlflow.start_run(run_name='winner'):
    mlflow.log_params(study.best_params)
    mlflow.log_param('data_md5', data_md5)
    mlflow.log_param('seed', SEED)
    mlflow.log_metric('val_rmse', study.best_value)
    mlflow.sklearn.log_model(winner, name='model', registered_model_name='ccpp-hgb')
print('registered ccpp-hgb with data_md5', data_md5[:12], '...')

## 5. Load it back by URI, and score the test set once

A registered model is loaded by a `models:/` URI, not by a file path. We fetch the
latest version, load it on a clean handle, and only now touch the locked test set. You
report this test number, not the best validation score, because the search optimized the
validation score. On this large, easy dataset the two land close together, and any small
difference here is mostly the winner training on more data than the cross-validation
folds rather than selection bias, which the module measured as negligible at full size.
On small data the validation score would instead be optimistically low, which is the
reason to report the test at all.

In [ ]:
from mlflow import MlflowClient
from sklearn.metrics import root_mean_squared_error

client = MlflowClient()
latest = max(int(v.version) for v in client.search_model_versions("name='ccpp-hgb'"))
uri = f'models:/ccpp-hgb/{latest}'
loaded = mlflow.sklearn.load_model(uri)

test_rmse = root_mean_squared_error(y_test, loaded.predict(X_test))
print(f'loaded {uri}')
print(f'best VALIDATION RMSE (search optimized) : {study.best_value:.3f} MW')
print(f'held-out TEST RMSE (reported once)      : {test_rmse:.3f} MW')

---

## Takeaway

The search optimized the validation score; the number you report is the test score,
computed once on the model you selected. Every trial is in the tracking store with its
parameters and metric, the winner is registered with its data hash and seed, and it
loads back by a `models:/` URI on any machine. That record of what you tried and why is
the deliverable. Assignment **A5** has you take one dataset from data to a defended model
choice with every run tracked, so it starts here.